# HeatUp — Production Notebook 1: Structure Preparation & AIMD

Runs the complete pre-stability workflow for a list of candidate materials:

| Step | What it does | Output |
|------|-------------|--------|
| 1 | MACE geometry relaxation | `relaxation/CONTCAR`, `energy.json` |
| 2 | Finite-displacement phonons | `phonons/dos.json`, `phonon.pdf` |
| 3 | Elastic tensor | `elastic/elastic_tensor.json` |
| 4 | AIMD supercell construction | `aimd/POSCAR` |
| 5 | NPT molecular dynamics | `aimd/<T>K/output.traj`, `analysis.json` |
| 6 | Anharmonic phonons (VACF→VDOS) | `aimd/<T>K/anharmonic_phonons/vdos.json` |

**All steps are idempotent.** Re-running this notebook skips steps whose outputs already exist.

**Expected runtime** (per material, single GPU):
- Relaxation: ~2–5 min
- Phonons: ~10–30 min (depends on supercell size)
- Elastic: ~2–5 min
- AIMD (30 ps): ~30–120 min (depends on system size)
- VACF/VDOS: ~30 s (post-processing)

---
**Configure paths in Section 1 only. Never edit the library modules.**

## 0 — Imports

In [1]:
import json
import os
import subprocess
import sys
import warnings
import shutil

import torch
import pandas as pd
from IPython.display import display, Image

# ── Active-learning library (from the SSE-AL project) ────────────────────────
# Ensure the project root is on sys.path so `from library.X import ...` works.
# Adjust PROJECT_ROOT in Section 1 if this notebook is not in the project root.
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from library.md_pipeline import (
    MACE_MODEL, NBLOCK, N_STEPS, TIMESTEP_FS, STEP_EQUIV,
    load_analysis, print_database_summary,
    run_md_simulation, scan_database,
)
from library.structure_pipeline import (
    AIMD_MIN_CELL_ANG, ELASTIC_DELTA, FMAX, PHONON_SUPERCELL,
    prepare_aimd_folders,
    run_relaxation, run_relaxation_subprocess,
    run_phonons,    run_phonons_subprocess,
    run_elastic,    run_elastic_subprocess,
    run_all,
)
from library.anharmonic_phonons import compute_anharmonic_phonons_for_sim

warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device         : {DEVICE}')
print(f'MACE model     : {MACE_MODEL}')
print(f'Timestep       : {TIMESTEP_FS} fs')
print(f'MD steps       : {N_STEPS}  ({N_STEPS * TIMESTEP_FS / 1000:.0f} ps)')
print(f'Equil. frames  : {STEP_EQUIV}  ({STEP_EQUIV * NBLOCK * TIMESTEP_FS / 1000:.0f} ps)')
print(f'Relaxation fmax: {FMAX} eV/Å')
print(f'Phonon sc      : {PHONON_SUPERCELL}')
print(f'Elastic delta  : {ELASTIC_DELTA}')
print(f'Min AIMD cell  : {AIMD_MIN_CELL_ANG} Å')

ModuleNotFoundError: No module named 'library'

## 1 — Configuration

**Edit only this cell.** All physics parameters are defined in
`library/md_pipeline.py` and `library/structure_pipeline.py`.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Root of the simulation database. Subdirectories are created automatically.
DATABASE_ROOT  = 'database'

# Root of the candidate POSCAR tree:
#   input/candidates/<material>/<symmetry>/POSCAR
CANDIDATES_DIR = 'input/candidates'

# ── MACE model path ───────────────────────────────────────────────────────────
# Absolute or relative path to the MACE model file.
# This overrides the default defined in library/md_pipeline.py.
# Set to None to use the library default.
MACE_MODEL_PATH = 'mace-mpa-0-medium.model'

# ── Run list ──────────────────────────────────────────────────────────────────
# Two modes: read from query file (active-learning workflow)
# or specify manually (first run, reruns, extra temperatures).
USE_QUERY_FILE = False     # True = read QUERY_FILE; False = use MANUAL_LIST
QUERY_FILE     = 'next_candidates.json'

# Manual list: each entry needs 'material', 'symmetry', 'temperature'.
# Optional: 'force_rerun' (bool) to redo an already-completed simulation.
MANUAL_LIST = [
    {'material': 'Li10Ge\(PS6\)2', 'symmetry': 'P4_2mc', 'temperature': 600},
]

# ── Step flags ────────────────────────────────────────────────────────────────
# Set to False to skip a step entirely (e.g. if you only want to (re)run MD).
RUN_RELAXATION = True
RUN_PHONONS    = True
RUN_ELASTIC    = True
RUN_AIMD_PREP  = True    # build the MD supercell
RUN_MD         = True    # run the NPT simulation
RUN_ANHARMONIC = True    # VACF→VDOS post-processing

# ── Hull temperature grid (passed to anharmonic phonon analysis) ───────────────
HULL_TEMPERATURES = list(range(0, 1501, 50))    # K

os.makedirs(DATABASE_ROOT, exist_ok=True)

# Override library model path if specified
if MACE_MODEL_PATH is not None:
    import library.md_pipeline as _mdp
    import library.structure_pipeline as _sp
    _mdp.MACE_MODEL = MACE_MODEL_PATH
    _sp.MACE_MODEL  = MACE_MODEL_PATH
    print(f'MACE model overridden: {MACE_MODEL_PATH}')

print('Configuration loaded.')

## 2 — Active parameters summary

In [ ]:
from library.md_pipeline import MACE_MODEL as _MM
from library.structure_pipeline import FMAX as _FM, PHONON_SUPERCELL as _PS

params = pd.DataFrame([
    ['MACE model',         _MM],
    ['MD timestep',        f'{TIMESTEP_FS} fs'],
    ['MD total steps',     f'{N_STEPS}  ({N_STEPS * TIMESTEP_FS / 1000:.0f} ps)'],
    ['MD write interval',  f'every {NBLOCK} steps  ({NBLOCK * TIMESTEP_FS / 1000 * 1000:.0f} fs/frame)'],
    ['Equilibration',      f'{STEP_EQUIV} frames  ({STEP_EQUIV * NBLOCK * TIMESTEP_FS / 1000:.1f} ps)'],
    ['Relaxation fmax',    f'{_FM} eV/Å'],
    ['Phonon supercell',   str(_PS)],
    ['Elastic delta',      str(ELASTIC_DELTA)],
    ['Min AIMD cell',      f'{AIMD_MIN_CELL_ANG} Å'],
    ['Compute device',     DEVICE],
], columns=['Parameter', 'Value'])

display(params.set_index('Parameter').style.set_caption('Active simulation parameters'))

## 3 — Build run list

In [ ]:
if USE_QUERY_FILE:
    if not os.path.exists(QUERY_FILE):
        raise FileNotFoundError(
            f'{QUERY_FILE} not found.\n'
            'Run 01_query.ipynb first, or set USE_QUERY_FILE = False.'
        )
    with open(QUERY_FILE) as fh:
        run_list = json.load(fh)
    print(f'Loaded {len(run_list)} entries from {QUERY_FILE}')
else:
    run_list = MANUAL_LIST
    print(f'Manual list: {len(run_list)} entries')

# Validate and build resolved entries
resolved = []
for entry in run_list:
    mat  = entry['material']
    sym  = entry['symmetry']
    temp = int(entry['temperature'])
    force = entry.get('force_rerun', False)

    # Look up POSCAR: prefer database/ then candidates/
    poscar_db   = os.path.join(DATABASE_ROOT,  mat, sym, 'POSCAR')
    poscar_cand = os.path.join(CANDIDATES_DIR, mat, sym, 'POSCAR')

    if os.path.exists(poscar_db):
        src = poscar_db
    elif os.path.exists(poscar_cand):
        src = poscar_cand
    else:
        print(f'  [MISSING] {mat}/{sym}: POSCAR not found in database/ or candidates/ — skipping')
        continue

    sym_dir = os.path.join(DATABASE_ROOT, mat, sym)
    os.makedirs(sym_dir, exist_ok=True)

    # Copy POSCAR to database if it came from candidates
    dst_poscar = os.path.join(sym_dir, 'POSCAR')
    if not os.path.exists(dst_poscar):
        shutil.copy(src, dst_poscar)
        print(f'  POSCAR copied  → {dst_poscar}')

    resolved.append({
        'material'   : mat,
        'symmetry'   : sym,
        'temperature': temp,
        'sym_dir'    : sym_dir,
        'force_rerun': force,
    })

print(f'\nResolved {len(resolved)} entries:')
for e in resolved:
    print(f"  {e['material']}/{e['symmetry']}  T={e['temperature']} K"
          + ('  [FORCE RERUN]' if e['force_rerun'] else ''))

## 4 — Current database state

In [ ]:
print_database_summary(DATABASE_ROOT)

## 5 — Structure preparation + AIMD simulation

For each resolved entry, runs in sequence:
1. Relaxation (BFGS with MACE)
2. Phonon DOS + band structure
3. Elastic tensor
4. AIMD supercell construction
5. NPT MD via isolated subprocess (CUDA memory released on exit)
6. Anharmonic VDOS from VACF

In [ ]:
NFAIL = 0
completed = []

for entry in resolved:
    mat     = entry['material']
    sym     = entry['symmetry']
    temp    = entry['temperature']
    sym_dir = entry['sym_dir']
    force   = entry['force_rerun']
    tag     = f'{mat}/{sym}  T={temp} K'

    print(f'\n{"=" * 65}')
    print(f'  {tag}')
    print(f'{"=" * 65}')

    # ── Step 1: Relaxation ────────────────────────────────────────────────────
    if RUN_RELAXATION:
        ret = run_relaxation_subprocess(sym_dir, device=DEVICE, force_rerun=force)
        if not ret:
            print(f'  [WARN] Relaxation failed for {tag} — continuing anyway')

    # ── Step 2: Phonons ───────────────────────────────────────────────────────
    if RUN_PHONONS:
        ret = run_phonons_subprocess(sym_dir, device=DEVICE, force_rerun=force)
        if not ret:
            print(f'  [WARN] Phonon calculation failed for {tag}')

    # ── Step 3: Elastic tensor ────────────────────────────────────────────────
    if RUN_ELASTIC:
        ret = run_elastic_subprocess(sym_dir, device=DEVICE, force_rerun=force)
        if not ret:
            print(f'  [WARN] Elastic calculation failed for {tag}')

    # ── Step 4: AIMD supercell preparation ───────────────────────────────────
    if RUN_AIMD_PREP:
        try:
            prepare_aimd_folders(sym_dir, temperatures=[temp])
        except Exception as exc:
            print(f'  [WARN] AIMD prep failed: {exc}')

    # ── Step 5: NPT MD via subprocess ─────────────────────────────────────────
    if RUN_MD:
        script = os.path.join(PROJECT_ROOT, 'library', 'run_single_md.py')
        cmd    = [
            sys.executable, script,
            sym_dir, str(temp),
            '--device', DEVICE,
        ]
        if force:
            cmd.append('--force')

        env = os.environ.copy()
        env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

        print(f'  Running MD via subprocess: {" ".join(cmd[2:])}')
        result = subprocess.run(cmd, env=env)
        if result.returncode != 0:
            print(f'  [FAIL] MD subprocess exited with code {result.returncode}')
            NFAIL += 1
            continue

    # ── Step 6: Anharmonic VDOS (VACF→VDOS) ──────────────────────────────────
    if RUN_ANHARMONIC:
        sim_dir = os.path.join(sym_dir, 'aimd', f'{temp}K')
        if os.path.exists(sim_dir):
            try:
                fe = compute_anharmonic_phonons_for_sim(
                    sim_dir,
                    temperatures = HULL_TEMPERATURES,
                    force_recompute = force,
                )
                if fe is not None:
                    print(f'  Anharmonic VDOS cached → {sim_dir}/anharmonic_phonons/')
                    contributions = fe.get('available_contributions', [])
                    if contributions:
                        print(f'  Free-energy contributions: {contributions}')
                else:
                    print(f'  [WARN] Anharmonic phonon analysis returned None for {tag}')
            except Exception as exc:
                print(f'  [WARN] Anharmonic phonons failed: {exc}')

    completed.append(tag)

print(f'\n{"-" * 50}')
print(f'Completed: {len(completed)}/{len(resolved)}')
if NFAIL:
    print(f'Failures:  {NFAIL}')
print(f'{"-" * 50}')

## 6 — Post-run database summary

In [ ]:
print_database_summary(DATABASE_ROOT)

## 7 — Quick diffusivity overview

Plots D (cm²/s) and σ (mS/cm) for all completed simulations in the database.

In [ ]:
import matplotlib.pyplot as plt

records = scan_database(DATABASE_ROOT)
if not records:
    print('No completed simulations found.')
else:
    rows = []
    for rec in records:
        for sp, vals in rec.get('diffusion', {}).items():
            D = vals.get('diffusivity_cm2s', 0)
            s = vals.get('conductivity_mScm', 0)
            if D > 1e-15:   # exclude zero-diffusivity entries
                rows.append({
                    'label'     : f"{rec['material']}/{rec['symmetry']}/{rec.get('temperature_K',0):.0f}K",
                    'species'   : sp,
                    'D_cm2s'    : D,
                    'sigma_mScm': s,
                })

    if rows:
        df = pd.DataFrame(rows).sort_values('sigma_mScm', ascending=False)
        display(df.head(20))

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        top = df.head(15)
        ax1.barh(top['label'] + '/' + top['species'], np.log10(top['D_cm2s'].clip(1e-20)),
                 color='steelblue', alpha=0.8)
        ax1.set_xlabel('log₁₀ D (cm²/s)')
        ax1.set_title('Diffusivity (top 15)')

        ax2.barh(top['label'] + '/' + top['species'], top['sigma_mScm'],
                 color='darkorange', alpha=0.8)
        ax2.set_xlabel('σ (mS/cm)')
        ax2.set_title('Ionic conductivity (top 15)')

        plt.tight_layout()
        plt.savefig('conductivity_overview.pdf', dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved: conductivity_overview.pdf')